<a href="https://colab.research.google.com/github/BemcrosfBrookes/COMP5046/blob/main/19322435-UserStory_7_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Task 7
# Settings and account backend 7.1  7.2  7.3

SESSION = {
    "logged_in": False,
    "email": None,
    "current_screen": "login",      # "login" "main" "settings"
    "notifications_on": True,       # single toggle for all app notifications on or off
    "default_notification_time": "08:00"   # simple text time eg "08:00"
}

PROFILE_STORE = {
    # Something like
    # "user_email": {"name": "...", "surname": "...", "email": "..."}
}

# 7.1
# notification toggle and exit or logout

def open_settings_screen(user_email):
    SESSION["logged_in"] = True
    SESSION["email"] = user_email
    SESSION["current_screen"] = "settings"


def exit_settings_to_main():
    if SESSION["logged_in"]:
        SESSION["current_screen"] = "main"


def logout_user():
    SESSION["logged_in"] = False
    SESSION["email"] = None
    SESSION["current_screen"] = "login"
    SESSION["notifications_on"] = True


def set_notification_toggle(from_html_toggle):
    SESSION["notifications_on"] = bool(from_html_toggle)


def load_notification_toggle_for_html():
    return {
        "notifications_on": SESSION["notifications_on"]
    }


def notifications_allowed():
    return SESSION.get("notifications_on", True)


# 7.2
# change and save account settings in the ReminderAppUserData document

def load_users_from_document(file_path):
    users = []

    try:
        with open(file_path, "r") as f:
            for line in f:
                line = line.strip()
                if line == "":
                    continue

                parts = line.split(",")
                if len(parts) < 4:
                    continue

                email = parts[0].strip()
                name = parts[1].strip()
                surname = parts[2].strip()
                password = parts[3].strip()

                users.append({
                    "email": email,
                    "name": name,
                    "surname": surname,
                    "password": password
                })
    except FileNotFoundError:
        return []

    return users


def save_users_to_document(file_path, users):
    with open(file_path, "w") as f:
        for user in users:
            line = "{},{},{},{}\n".format(
                user["email"],
                user["name"],
                user["surname"],
                user["password"]
            )
            f.write(line)


def update_account_in_document(current_email,
                               file_path,
                               new_email=None,
                               new_name=None,
                               new_surname=None,
                               new_password=None):
    users = load_users_from_document(file_path)

    for user in users:
        if user["email"] == current_email:
            if new_email is not None and new_email.strip() != "":
                user["email"] = new_email.strip()

            if new_name is not None and new_name.strip() != "":
                user["name"] = new_name.strip()

            if new_surname is not None and new_surname.strip() != "":
                user["surname"] = new_surname.strip()

            if new_password is not None and new_password.strip() != "":
                user["password"] = new_password.strip()

            break

    save_users_to_document(file_path, users)

    # keep SESSION email in sync if needed (only really matters when integrated)
    if SESSION.get("logged_in") and SESSION.get("email") == current_email:
        if new_email is not None and new_email.strip() != "":
            SESSION["email"] = new_email.strip()

    return users


# 7.3
# default time, logout to login, profile data

def set_default_notification_time(from_html_time_value):
    if from_html_time_value is None:
        return

    time_text = str(from_html_time_value).strip()
    if time_text == "":
        return

    SESSION["default_notification_time"] = time_text


def get_default_notification_time_for_html():
    return SESSION["default_notification_time"]


def logout_and_go_to_login():
    SESSION["logged_in"] = False
    SESSION["email"] = None
    SESSION["current_screen"] = "login"


def save_profile_details(email, name, surname):
    PROFILE_STORE[email] = {
        "name": name,
        "surname": surname,
        "email": email
    }


def get_profile_for_logged_in_user():
    if not SESSION["logged_in"]:
        return None

    user_email = SESSION["email"]
    data = PROFILE_STORE.get(user_email)

    if data is None:
        return None

    return {
        "name": data.get("name", ""),
        "surname": data.get("surname", ""),
        "email": data.get("email", "")
    }